In [ ]:
import ast
from collections import Counter
from pathlib import Path
from typing import Iterable, Optional, Union

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

In [ ]:
COLAB_DIR = Path('/content/drive/MyDrive')
KAGGLE_DIR = Path('/kaggle/input')
LOCAL_DIR = Path('../')

try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    import kaggle_secrets

    IN_KAGGLE = True
except ImportError:
    IN_KAGGLE = False

if IN_COLAB:
    DATA_DIR = COLAB_DIR / 'data'
    MODELS_DIR = COLAB_DIR / 'models'
elif IN_KAGGLE:
    DATA_DIR = KAGGLE_DIR
    MODELS_DIR = KAGGLE_DIR
else:
    DATA_DIR = LOCAL_DIR / 'data'
    MODELS_DIR = LOCAL_DIR / 'models'

In [ ]:
TWEETS_CSV = DATA_DIR / 'tweets-dataset-for-cyberattack-detection' / 'tweets_final.csv'

In [ ]:
filter_df = pd.read_csv(filepath_or_buffer=TWEETS_CSV)
filter_df.dropna(subset=['relevant'])

print(f"There are {filter_df.shape[0]} tweets")
filter_df.head()

In [ ]:
X_train_val, X_test = train_test_split(
    filter_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=filter_df['relevant']
)

X_train_val.shape, X_test.shape

In [ ]:
with open(DATA_DIR / 'tweets-dataset' / 'filter_train.csv', 'wb') as f:
    X_train_val.to_csv(path_or_buf=f)

with open(DATA_DIR / 'tweets-dataset' / 'filter_test.csv', 'wb') as f:
    X_test.to_csv(path_or_buf=f)

In [ ]:
X_train_val = pd.read_csv(filepath_or_buffer=DATA_DIR / 'tweets-dataset' / 'filter_train.csv')
X_test = pd.read_csv(filepath_or_buffer=DATA_DIR / 'tweets-dataset' / 'filter_test.csv')

X_train_val.shape, X_test.shape

In [ ]:
def build_tree(categories: list[dict]) -> dict:
    """
    Build a tree-like structure (nested dictionary) from category labels.

    Parameters
    ----------
    categories
        A list of categories, where each category has a 'label' key that contains a path-like string.

    Returns
    -------
    tree
        A nested dictionary representing the tree structure.
    """
    tree = {}

    for category in categories:
        current = tree

        for part in category['label'].strip('/').split('/'):
            current = current.setdefault(part, {})

    return tree
    
def extract_keys(d: Union[dict, object], path: Optional[Iterable] = None) -> list:
    """
    Recursively extract keys from a dictionary, building paths as a list.

    Parameters
    ----------
    d
        The dictionary to extract keys from.
    path
        A list to accumulate the path, default is None.

    Returns
    -------
    path
        A list of paths representing keys in the dictionary.
    """
    if path is None:
        path = []

    if isinstance(d, dict):
        for key, value in d.items():
            path = extract_keys(d=value, path=path + [key])
    else:
        path = [d]

    return path

def map_targets(watson_list, fix_targets):
    targets = set(fix_targets.keys()) & set(watson_list)
    mapped_targets = {fix_targets[category] for category in targets}
    return list(mapped_targets) if mapped_targets else ['other']

In [ ]:
FIX_TARGETS = {
    'computer security': 'computer security',
    'operating systems': 'operating systems',
    'software': 'software',
    'programming languages': 'software',
    'hardware': 'hardware',
    'electronic components': 'hardware',
    'networking': 'networking',
    'internet technology': 'networking',
    'network security': 'network security',
    'antivirus and malware': 'antivirus and malware',
    'mac os': 'mac os',
    'windows': 'windows',
    'unix': 'unix',
    'linux': 'linux',
    'databases': 'databases',
    'computer': 'computer',
    'computer components': 'computer components',
    'computer networking': 'computer networking',
    'servers': 'servers',
    'portable computer': 'portable computer',
    'desktop computer': 'desktop computer',
    'router': 'router',
    'wireless technology': 'wireless technology'
}

categories_df = (
    pd.read_csv(filepath_or_buffer=TWEETS_CSV)
    .assign(
        tweet=lambda df: df['tweet'].apply(func=ast.literal_eval),
        watson=lambda df: df['watson'].apply(func=ast.literal_eval)
        .apply(func=lambda x: x.get('categories', []))
        .apply(func=build_tree),
        watson_list=lambda df: df['watson'].apply(func=extract_keys),
        target=lambda df: df['watson_list'].apply(func=map_targets, args=(FIX_TARGETS,))
    )
    .query(expr='relevant == True')
)
mask = ['other' in target_i for target_i in categories_df['target']]
categories_df_temp = categories_df[mask]
categories_df_temp = categories_df_temp[categories_df_temp['relevant'] == False]
categories_df = categories_df.loc[list(set(categories_df.index) - set(categories_df_temp.index))]
categories_df = categories_df.reset_index(drop=True)

print(f"There are {categories_df.shape[0]} tweets")
categories_df.head()

In [ ]:
class_freq = Counter([item for sublist in categories_df['target'] for item in sublist])

print("Target Frequencies:")
for target, freq in class_freq.most_common():
    print(f"{target}: {freq}")

_, ax = plt.subplots(figsize=(18, 6))
pd.Series(data=class_freq).sort_values(ascending=False).plot(kind='bar', color='skyblue')
plt.title('Distribution of Targets')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.show()

In [ ]:
target_freq = Counter(tuple(item) for item in categories_df['target'])
cut = len(target_freq)

print("Target Frequencies:")
for target, freq in target_freq.most_common()[:cut]:
    print(f"{target}: {freq}")

_, ax = plt.subplots(figsize=(18, 6))

(pd.Series(data={str(k): v for k, v in target_freq.most_common()[:cut]})
 .sort_values(ascending=False)
 .plot(kind='bar', color='skyblue'))

plt.title('Distribution of Targets')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.show()
print(f'Number of targets to subsample:\t{sum([1 if v < 10 else 0 for v in target_freq.values()])}')

In [ ]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(y=categories_df['target'])
labels_map, _ = np.unique(ar=y, axis=0, return_inverse=True)

In [ ]:
target_freq = Counter(tuple(item) for item in categories_df['target'])

print("Target Frequencies:")
for target, freq in target_freq.most_common():
    print(f"{target}: {freq}")

_, ax = plt.subplots(figsize=(18, 6))
(pd.Series(data={str(k): v for k, v in target_freq.items()})
 .sort_values(ascending=False)
 .plot(kind='bar', color='skyblue'))
plt.title('Distribution of Targets')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.show()

valid_targets = {target for target, freq in target_freq.items() if freq >= 5}

# Filter the DataFrame to keep only valid targets
categories_df = categories_df[categories_df['target'].apply(lambda x: tuple(x) in valid_targets)]

# Reset the index (optional)
categories_df = categories_df.reset_index(drop=True)

# Print the updated target frequencies
updated_target_freq = Counter(tuple(item) for item in categories_df['target'])
print("Updated Target Frequencies:")
for target, freq in updated_target_freq.most_common():
    print(f"{target}: {freq}")

_, ax = plt.subplots(figsize=(18, 6))
(pd.Series(data={str(k): v for k, v in updated_target_freq.items()})
 .sort_values(ascending=False)
 .plot(kind='bar', color='skyblue'))
plt.title('Distribution of Targets')
plt.xlabel('Target')
plt.ylabel('Frequency')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.show()

In [ ]:
_, y_mcp = np.unique(ar=mlb.transform(categories_df['target']), axis=0, return_inverse=True)
X_train_val, X_test = train_test_split(
    categories_df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=y_mcp
)

In [ ]:
with open(DATA_DIR / 'tweets-dataset' / 'categories_train.csv', 'wb') as f:
    X_train_val.to_csv(path_or_buf=f)

with open(DATA_DIR / 'tweets-dataset' / 'categories_test.csv', 'wb') as f:
    X_test.to_csv(path_or_buf=f)